In [2]:
!pip install faiss-cpu groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 12.2 MB/s eta 0:00:00


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import random

import torch
import numpy as np
import pandas as pd
from groq import Groq
from sentence_transformers import util, SentenceTransformer
from google.colab import userdata
from transformers import AutoTokenizer
import faiss


device = "cuda" if torch.cuda.is_available() else "cpu"

# Import texts and embedding df
text_chunks_and_embedding_df = pd.read_csv("Data/embedding.csv")

# Convert embedding column back to np.array (it got converted to string when it got saved to CSV)
text_chunks_and_embedding_df["embedding"] = text_chunks_and_embedding_df["embedding"].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))

# Convert texts and embedding df to list of dicts
pages_and_chunks = text_chunks_and_embedding_df.to_dict(orient="records")

# Convert embeddings to torch tensor and send to device (note: NumPy arrays are float64, torch tensors are float32 by default)
embeddings = torch.tensor(np.array(text_chunks_and_embedding_df["embedding"].tolist()), dtype=torch.float32).to(device)
embeddings.shape

torch.Size([938, 768])

In [4]:

embedding_model = SentenceTransformer(model_name_or_path="all-mpnet-base-v2",
                                      device=device) # choose the device to load the model to

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:

groq_api = userdata.get('groq')
#import groq

In [6]:

client = Groq(api_key=groq_api,)

chat_completion = client.chat.completions.create(
    messages=[{"role": "user", "content": "Explain Groq speed"}],
    model="llama-3.3-70b-versatile",
)
print(chat_completion.choices[0].message.content)


Groq is a high-performance computing architecture designed for large-scale artificial intelligence (AI) and machine learning (ML) workloads. It's particularly focused on matrix multiplication, which is a fundamental operation in many AI algorithms.

Groq speed refers to the processing speed and efficiency of the Groq architecture in executing these complex matrix operations. Here are some key factors that contribute to Groq's speed:

1. **Tensor Processing Units (TPUs)**: Groq's architecture is based on custom-designed TPUs, which are optimized for matrix multiplication and other AI-related operations. These TPUs provide a significant boost in performance compared to traditional CPUs or GPUs.
2. **High-Bandwidth Memory**: Groq's system features high-bandwidth memory, which enables fast data transfer between the TPUs and the main memory. This reduces memory access latency and increases overall system performance.
3. **Low-Latency Interconnects**: Groq's architecture includes low-latency

In [7]:
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

{'input_ids': [1, 1824, 349, 399, 2377, 28804], 'attention_mask': [1, 1, 1, 1, 1, 1]}


In [11]:
import faiss

In [14]:
def prompt_formatter(query: str,
                     context: str) -> str:
    """
    Augments query with text-based context from context_items.
    """
    # Join context items into one dotted paragraph
    #context = "- " + "\n- ".join([item["sentence_chunk"] for item in context_items])

    # Create a base prompt with examples to help the model
    # Note: this is very customizable, I've chosen to use 3 examples of the answer style we'd like.
    # We could also write this in a txt file and import it in if we wanted.
    base_prompt = """You must answer strictly using the provided context.

Context:
{context}

Question:
{query}

Rules:
	•	Every part of the answer MUST be directly supported by the source
	•	Do NOT infer or assume relationships (e.g., revenue ≠ margin unless explicitly stated)
	•	If any part of the question is not explicitly answered → say “Not explicitly stated in context”
	•	Include numerical values if present (%, basis points, etc.)
	•	Ensure answer is complete (cause + effect if applicable)
  •	In the source mention which part of the context you are referring to

Return JSON:
{{
“answer”: “…”,
“confidence”: “high/medium/low”,
“source”:}} """

    # Update base prompt with context items and query
    base_prompt = base_prompt.format(context=context, query=query)

    # Create prompt template for instruction-tuned model
    dialogue_template = [
        {"role": "user",
        "content": base_prompt}
    ]

    # Apply the chat template
    prompt = tokenizer.apply_chat_template(conversation=dialogue_template,
                                          tokenize=False,
                                          add_generation_prompt=True)
    return prompt

In [27]:
Q = ["Which vertical contributed the highest revenue, and how does its growth compare to overall company growth?",
	"List the top 3 geographies by revenue share and identify which one had the slowest growth.",
	"TCS mentions investments in AI and cloud. Which verticals are most likely to benefit from both, based on deal wins and partnerships described?",
	"Despite positive revenue growth, margins declined. Identify all contributing factors mentioned and classify them into controllable vs uncontrollable.",
	"If revenue grew 6% and employee cost % declined, what other cost components likely drove margin pressure?",
	"Using the ratio table, determine whether TCS became more or less operationally efficient YoY. Justify with 2–3 ratios.",
	"The report claims strong operational discipline. Identify any metrics that contradict this statement.",
	"Do the hiring trends align with the stated focus on cost optimization? Provide evidence.",
	"Which client contributed the highest revenue growth in FY 2025?",
	"Explain how the ‘Perpetually Adaptive Enterprise’ strategy improved operating margin.",
	"Combine currency impact, geography mix, and vertical performance to explain overall revenue growth drivers.",
	"If GBP had depreciated instead of appreciating, estimate the qualitative impact on reported growth."]
Q = ["How's TCS's strategy better than Infosys and IBM"]

In [28]:
embeddings = np.array(text_chunks_and_embedding_df.embedding)
embeddings = np.vstack(embeddings)
embeddings = embeddings.astype('float32')
faiss.normalize_L2(embeddings)
index_flat = faiss.IndexFlatIP(embeddings.shape[1])  # Inner product (= cosine sim for normalized vectors)

# FAISS requires float32 and L2-normalized vectors for cosine similarity

index_flat.add(embeddings)
print(f"Flat Index: {index_flat.ntotal} vectors indexed (exact search)")


Flat Index: 938 vectors indexed (exact search)


In [29]:
# Test query

for q in Q:
  test_query = q
  query_emb = embedding_model.encode([test_query]).astype('float32')
  faiss.normalize_L2(query_emb)

  K = 8  # Top-5 results

  print(f"Query: \"{test_query}\"")
  print(f"{'=' * 70}")

  for idx_name, idx in [("Flat (Exact)", index_flat),
                        ]:

      scores, indices = idx.search(query_emb, K)

  context = "\n".join([str(i+1)+'. '+text for i,text in enumerate(list(text_chunks_and_embedding_df.loc[indices[0],'sentence_chunk']))])

  client = Groq(api_key=groq_api,)

  chat_completion = client.chat.completions.create(
      messages=[{"role": "user", "content": prompt_formatter(test_query,context)}],
      model="openai/gpt-oss-120b",
  )
  print(chat_completion.choices[0].message.content)


Query: "How's TCS's strategy better than Infosys and IBM"
{
  "answer": "Not explicitly stated in context",
  "confidence": "high",
  "source": null
}
